# TorGen: DETR-CVAE for Tornado Outbreak Generation

Install the package, mount Drive, and train.

In [ ]:
!pip install -q git+https://github.com/jcaramichaellehigh/TorGen.git
!pip show torgen | grep -E "^(Version|Location)"
!python -c "
import json, urllib.request, importlib.metadata
dist = importlib.metadata.distribution('torgen')
du = dist.read_text('direct_url.json')
if du:
    sha = json.loads(du).get('vcs_info', {}).get('commit_id', '')
    if sha:
        try:
            url = f'https://api.github.com/repos/jcaramichaellehigh/TorGen/commits/{sha}'
            resp = json.loads(urllib.request.urlopen(url).read())
            msg = resp['commit']['message'].split('\n')[0]
            print(f'{sha[:8]} {msg}')
        except Exception:
            print(f'{sha[:8]} (could not fetch title)')
    else:
        print('No commit hash found')
else:
    print('Installed from non-git source')
"
!python -c "
from torgen.model.cvae import TorGenCVAE
from torgen.loss.hungarian import HungarianMatchingLoss
from torgen.eval.evaluate import run_evaluation
print('torgen v2 imported successfully')
"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Optional: experiment tracking
# Sign up at wandb.ai, get API key from wandb.ai/authorize
try:
    import wandb
    wandb.login()
except ImportError:
    print("wandb not installed, skipping experiment tracking")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from torgen.training import TrainConfig, train

config = TrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints",
    ef_weight_power=0.5,  # 0 = uniform, 0.5 = sqrt IF, 1 = full inverse-frequency
)

trainer = train(config)

In [ ]:
print(f"Epochs trained: {trainer.epoch}")
print(f"Final train loss: {trainer.train_losses[-1]:.4f}")
print(f"Final val loss: {trainer.val_losses[-1]:.4f}")
print(f"Best val loss: {trainer.best_val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(trainer.train_losses) + 1), trainer.train_losses, label="Train")
ax.plot(range(1, len(trainer.val_losses) + 1), trainer.val_losses, label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("TorGen Training Curves")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
import json
import os

eval_dir = os.path.join(config.checkpoint_dir, "eval")
summary_path = os.path.join(eval_dir, "summary.json")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        results = json.load(f)
    print("Evaluation Results:")
    for k, v in results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
else:
    print("No evaluation results found (evaluation runs automatically after training)")